# Script DataLoad

# Imports

In [47]:
from faker import Faker 
import psycopg2
from dotenv import load_dotenv
import pandas as pd
import random
import os

fake = Faker("pt_BR")

# Conexão

In [48]:
load_dotenv()

conn = psycopg2.connect(
    host=os.getenv("DB_HOST"),
    port=os.getenv("DB_PORT"),
    database=os.getenv("DB_NAME"),
    user=os.getenv("DB_USER"),
    password=os.getenv("DB_PASSWORD"),
    sslmode=os.getenv("DB_SSLMODE")
)

cursor = conn.cursor()

# Resetando o banco

In [ ]:
conn.rollback()

cursor.execute("""
TRUNCATE TABLE
tb_auditoria_transacao,
tb_carga_mensal_item,
tb_carga_mensal,
tb_transacao,
tb_estabelecimento,
tb_categoria_mcc,
tb_saldo_bolso,
tb_tipo_bolso,
tb_cartao,
tb_colaborador,
tb_empresa,
tb_grupo_empresarial
RESTART IDENTITY CASCADE;
""")

conn.commit()

print("Banco totalmente resetado!")

Banco resetado!


# INSERT Tipos de bolso (fixos)

In [29]:
tipos_bolso = [
    ("FOOD", "Vale Alimentação"),
    ("MEAL", "Vale Refeição"),
    ("MOBILITY", "Vale Mobilidade"),
    ("CULTURE", "Vale Cultura")
]

for codigo, descricao in tipos_bolso:
    cursor.execute("""
        INSERT INTO tb_tipo_bolso (codigo, descricao)
        VALUES (%s, %s)
    """, (codigo, descricao))

conn.commit()

print("Tipos de bolso criados!")

Tipos de bolso criados!


# Grupo empresarial

In [30]:
for _ in range(10):
    cursor.execute("""
        INSERT INTO tb_grupo_empresarial (nome, cnpj_raiz, ativo)
        VALUES (%s, %s, %s)
    """, (
        fake.company(),
        fake.random_number(digits=14, fix_len=True),
        True
    ))

conn.commit()

print("Grupos criados!")

Grupos criados!


# Empresas

In [31]:
cursor.execute("SELECT id_grupo FROM tb_grupo_empresarial")
grupos = cursor.fetchall()

for _ in range(40):
    cursor.execute("""
        INSERT INTO tb_empresa (id_grupo, nome, cnpj, ativa)
        VALUES (%s, %s, %s, %s)
    """, (
        random.choice(grupos)[0],
        fake.company(),
        fake.random_number(digits=14, fix_len=True),
        True
    ))

conn.commit()

print("Empresas criadas!")

Empresas criadas!


# Colaboradores

In [32]:
cursor.execute("SELECT id_empresa FROM tb_empresa")
empresas = cursor.fetchall()

if not empresas:
    raise Exception("Nenhuma empresa encontrada")

fake.unique.clear()

for i in range(200):

    cpf = fake.unique.random_number(digits=11, fix_len=True)

    cursor.execute("""
        INSERT INTO tb_colaborador (
            id_empresa,
            nome,
            cpf,
            matricula,
            data_admissao,
            status
        )
        VALUES (%s, %s, %s, %s, %s, %s)
    """, (
        random.choice(empresas)[0],
        fake.name(),
        cpf,
        f"MAT{i:05}",
        fake.date_between(start_date='-5y', end_date='today'),
        "ATIVO"
    ))

conn.commit()

print("Colaboradores criados com segurança!")

Colaboradores criados com segurança!


# Cartões

In [33]:
cursor.execute("SELECT id_colaborador FROM tb_colaborador")
colaboradores = cursor.fetchall()

for c in colaboradores:
    cursor.execute("""
        INSERT INTO tb_cartao (
            id_colaborador, numero_tokenizado, status, data_emissao, data_validade
        )
        VALUES (%s, %s, %s, %s, %s)
    """, (
        c[0],
        fake.uuid4(),
        "ATIVO",
        fake.date_this_year(),
        fake.date_between(start_date='today', end_date='+5y')
    ))

conn.commit()

print("Cartões criados!")

Cartões criados!


# Saldos

In [34]:
cursor.execute("SELECT id_cartao FROM tb_cartao")
cartoes = cursor.fetchall()

cursor.execute("SELECT id_tipo_bolso FROM tb_tipo_bolso")
tipos = cursor.fetchall()

for cartao in cartoes:
    for tipo in tipos:
        cursor.execute("""
            INSERT INTO tb_saldo_bolso (
                id_cartao, id_tipo_bolso, saldo_atual
            )
            VALUES (%s, %s, %s)
        """, (
            cartao[0],
            tipo[0],
            round(random.uniform(100, 2000), 2)
        ))

conn.commit()

print("Saldos criados!")

Saldos criados!


# Adicionando mcc's

In [36]:
mccs = [
    ("5411", "Supermercados", 1),
    ("5812", "Restaurantes", 2),
    ("4111", "Transporte", 3),
    ("7832", "Cinema", 4),
    ("5462", "Padaria", 1),
    ("5814", "Fast Food", 2),
    ("4121", "Taxi", 3),
    ("7922", "Teatro", 4)
]

for mcc, desc, bolso in mccs:
    cursor.execute("""
        INSERT INTO tb_categoria_mcc (
            mcc, descricao, id_tipo_bolso, ativa
        )
        VALUES (%s, %s, %s, %s)
    """, (mcc, desc, bolso, True))

conn.commit()

# Estabelecimento

In [37]:
cursor.execute("SELECT id_categoria_mcc FROM tb_categoria_mcc")
cats = cursor.fetchall()

ufs = ["SP", "RJ", "MG", "PR", "SC"]

for _ in range(80):
    cursor.execute("""
        INSERT INTO tb_estabelecimento (
            nome, cnpj, id_categoria_mcc, cidade, uf
        )
        VALUES (%s, %s, %s, %s, %s)
    """, (
        fake.company(),
        fake.random_number(digits=14, fix_len=True),
        random.choice(cats)[0],
        fake.city(),
        random.choice(ufs)
    ))

conn.commit()

print("Estabelecimentos criados!")

Estabelecimentos criados!


# Transações

In [ ]:
import random
from decimal import Decimal

conn.rollback()

cursor.execute("""
SELECT 
    c.id_cartao, 
    sb.id_tipo_bolso, 
    sb.saldo_atual
FROM tb_cartao c
JOIN tb_saldo_bolso sb 
    ON sb.id_cartao = c.id_cartao
""")

cartoes = cursor.fetchall()

cursor.execute("""
SELECT 
    e.id_estabelecimento, 
    cm.id_tipo_bolso
FROM tb_estabelecimento e
JOIN tb_categoria_mcc cm 
    ON cm.id_categoria_mcc = e.id_categoria_mcc
""")

estabs = cursor.fetchall()

criados = 0
tentativas = 0

while criados < 500 and tentativas < 3000:

    tentativas += 1

    cartao, tipo, saldo = random.choice(cartoes)

    validos = [e for e in estabs if e[1] == tipo]
    if not validos:
        continue

    est = random.choice(validos)

    if float(saldo) <= 5:
        continue

    saldo_float = float(saldo)

    valor = round(
        random.uniform(5, min(250, saldo_float * 0.7)),
        2
    )

    try:
        cursor.execute("""
            INSERT INTO tb_transacao (
                id_cartao,
                id_estabelecimento,
                id_tipo_bolso,
                valor,
                status,
                usuario_registro
            )
            VALUES (%s, %s, %s, %s, %s, %s)
        """, (
            cartao,
            est[0],
            tipo,
            valor,
            "APROVADA",
            "seed"
        ))

        criados += 1

    except Exception as e:
        conn.rollback()
        continue

conn.commit()

print(f"Transações criadas: {criados}")
print(f"Tentativas: {tentativas}")

🔥 Transações criadas: 500
⚡ Tentativas: 502


In [ ]:
conn.rollback()

import random

cursor.execute("""
SELECT c.id_cartao, sb.id_tipo_bolso, sb.saldo_atual
FROM tb_cartao c
JOIN tb_saldo_bolso sb ON sb.id_cartao = c.id_cartao
""")

cartoes = cursor.fetchall()

cursor.execute("""
SELECT e.id_estabelecimento, cm.id_tipo_bolso
FROM tb_estabelecimento e
JOIN tb_categoria_mcc cm 
    ON cm.id_categoria_mcc = e.id_categoria_mcc
""")

estabs = cursor.fetchall()

criados = 0
tentativas = 0

while criados < 500 and tentativas < 3000:

    tentativas += 1

    cartao, tipo, saldo = random.choice(cartoes)

    validos = [e for e in estabs if e[1] == tipo]
    if not validos:
        continue

    est = random.choice(validos)

    if float(saldo) <= 5:
        continue

    saldo_float = float(saldo)

    valor = round(
        random.uniform(5, min(250, saldo_float * 0.7)),
        2
    )

    try:
        cursor.execute("""
            INSERT INTO tb_transacao (
                id_cartao,
                id_estabelecimento,
                id_tipo_bolso,
                valor,
                status,
                usuario_registro
            )
            VALUES (%s, %s, %s, %s, %s, %s)
        """, (
            cartao,
            est[0],
            tipo,
            valor,
            "APROVADA",
            "seed"
        ))

        criados += 1

    except Exception:
        conn.rollback()
        continue

conn.commit()

print(f"Transações criadas: {criados}")
print(f"Tentativas: {tentativas}")

🔥 Transações criadas: 500
⚡ Tentativas: 503


In [ ]:
conn.rollback()

import random

cursor.execute("""
SELECT c.id_cartao, sb.id_tipo_bolso, sb.saldo_atual
FROM tb_cartao c
JOIN tb_saldo_bolso sb 
    ON sb.id_cartao = c.id_cartao
""")

cartoes = cursor.fetchall()

cursor.execute("""
SELECT e.id_estabelecimento, cm.id_tipo_bolso
FROM tb_estabelecimento e
JOIN tb_categoria_mcc cm 
    ON cm.id_categoria_mcc = e.id_categoria_mcc
""")

estabs = cursor.fetchall()

criados = 0
tentativas = 0

while criados < 500 and tentativas < 3000:

    tentativas += 1

    cartao, tipo, saldo = random.choice(cartoes)

    validos = [e for e in estabs if e[1] == tipo]
    if not validos:
        continue

    est = random.choice(validos)

    if float(saldo) <= 5:
        continue

    saldo_float = float(saldo)

    limite = min(250, saldo_float * 0.7)

    valor = round(random.uniform(5, limite), 2)

    try:
        cursor.execute("""
            INSERT INTO tb_transacao (
                id_cartao,
                id_estabelecimento,
                id_tipo_bolso,
                valor,
                status,
                usuario_registro
            )
            VALUES (%s, %s, %s, %s, %s, %s)
        """, (
            cartao,
            est[0],
            tipo,
            valor,
            "APROVADA",
            "seed"
        ))

        criados += 1

    except Exception:
        conn.rollback()
        continue

conn.commit()

print(f"Transações criadas: {criados}")
print(f"Tentativas: {tentativas}")

🔥 Transações criadas: 500
⚡ Tentativas: 502


# Finalização

In [ ]:
cursor.close()
conn.close()

print("Seed concluído com sucesso!")